# Kiểm chứng generator drift nhỏ
Đọc cấu hình và kết quả Spark đã hoàn tất. Notebook này không sinh lại nguồn hay huấn luyện lại model. Hai kịch bản giữ nguyên sự cố; chỉ số đo cảm biến đổi theo cấu hình.

In [1]:
"""Read the exact sensor simulation configuration used by the native Spark verification."""
from pathlib import Path
import json, os, hashlib, html
from IPython.display import display, HTML
configuration_path = Path("generator/configs/sensor-drift-small.json")
configuration = json.loads(configuration_path.read_text())
configuration

{'simulation': {'seed': 42,
  'start_ts': '2026-10-01T00:00:00Z',
  'duration_hours': 48,
  'machine_count': 20,
  'sample_seconds': 60},
 'sensor_drift': {'start_ts': '2026-10-02T00:00:00Z',
  'ramp_seconds': 3600,
  'temperature_offset_c': 20.0,
  'vibration_multiplier': 1.8}}

In [2]:
"""Read sealed measurements and show source/label/readback checks without rerunning Spark."""
report_path = Path(os.environ["IOT_SENSOR_DRIFT_REPORT"])
report = json.loads(report_path.read_text())
assert report["passed"] and report["config"] == configuration
{"report_sha256": hashlib.sha256(report_path.read_bytes()).hexdigest(), "unique_source_rows_per_scenario": report["unique_telemetry_rows_per_scenario"], "joined_rows_per_scenario": report["joined_rows_per_scenario"], "labels_unchanged": report["labels_unchanged"], "all_persisted_values_readback_equal": report["all_persisted_values_readback_equal"], "sensor_effect_check": report["sensor_effect_check"]}

{'report_sha256': '9620b4913c4a5cd197c8aa3bb42af705fb062c6b92986780c802bceacabdfeda',
 'unique_source_rows_per_scenario': 57600,
 'joined_rows_per_scenario': 10373,
 'labels_unchanged': True,
 'all_persisted_values_readback_equal': True,
 'sensor_effect_check': {'max_temperature_shift_error': 4.263256414560601e-14,
  'max_vibration_scale_error': 0.0004000000000026205,
  'rows': 4828}}

In [3]:
"""Display genuine merged id/label/feature rows collected by the verified Spark job."""
columns = ["id", "label", "machine_id", "event_timestamp", "temperature_mean_10m", "vibration_max_10m"]
for scenario in ["baseline", "drift"]:
    rows = report["samples"][scenario][:4]
    header = "".join("<th>" + html.escape(name) + "</th>" for name in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(row[name])) + '</td>' for name in columns) + '</tr>' for row in rows)
    display(HTML("<h3>" + scenario + "</h3><table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))
assert [row["id"] for row in report["samples"]["baseline"]] == [row["id"] for row in report["samples"]["drift"]]
assert [row["label"] for row in report["samples"]["baseline"]] == [row["label"] for row in report["samples"]["drift"]]

id,label,machine_id,event_timestamp,temperature_mean_10m,vibration_max_10m
bb00d4685ed8db56ee1c79f3cf2f9927304b3a74ff8b4c8e751ccea4581dfac9,0,P-005,2026-10-02T10:45:00Z,56.588800000000006,3.279
d685738189abea35fb58b63b26523118c17c7c819156b071d73df85e3b6d6d76,0,P-011,2026-10-02T08:20:00Z,56.03912499999999,5.655
b38457f533676234faf8ff6e6913c6f82ee6bbd2eb4d23e966934e242e69019e,0,P-020,2026-10-02T21:15:00Z,59.34700000000001,5.887
dad441e92dd1eb54236b3feaebc64767b06156c1ef89e3d8bc135df3bc49e787,0,P-017,2026-10-02T22:10:00Z,70.215,6.418


id,label,machine_id,event_timestamp,temperature_mean_10m,vibration_max_10m
bb00d4685ed8db56ee1c79f3cf2f9927304b3a74ff8b4c8e751ccea4581dfac9,0,P-005,2026-10-02T10:45:00Z,76.5888,5.902
d685738189abea35fb58b63b26523118c17c7c819156b071d73df85e3b6d6d76,0,P-011,2026-10-02T08:20:00Z,76.039125,10.179
b38457f533676234faf8ff6e6913c6f82ee6bbd2eb4d23e966934e242e69019e,0,P-020,2026-10-02T21:15:00Z,79.34700000000001,10.597
dad441e92dd1eb54236b3feaebc64767b06156c1ef89e3d8bc135df3bc49e787,0,P-017,2026-10-02T22:10:00Z,90.215,11.552
